# ARC Prize 2026 — Public Submission Notebook
**Approach**: Compositional Test-Time Program Synthesis with Late Color Binding

Este notebook é auto-contido e gera o arquivo `submission.json` exigido pela competição ARC-AGI-2, além de servir como o **Attached Public Notebook** oficial para o **ARC Prize 2026 Paper Track**.

In [ ]:
import json
from pathlib import Path
import numpy as np

# Caminho padrão no ambiente do Kaggle para o ARC-AGI-2
DATA_DIR = Path('/kaggle/input/arc-prize-2026-arc-agi-2')
TEST_CHALLENGES_PATH = DATA_DIR / 'arc-agi_test_challenges.json'
SUBMISSION_PATH = Path('submission.json')

In [ ]:
# Primitivas e Síntese de Programas em Tempo de Teste
def identity(g): return g.copy()
def rot90(g): return np.rot90(g, 1)
def rot180(g): return np.rot90(g, 2)
def rot270(g): return np.rot90(g, 3)
def flip_h(g): return np.fliplr(g)
def flip_v(g): return np.flipud(g)
def transpose(g): return g.T.copy()

def crop_nonzero(g):
    coords = np.argwhere(g != 0)
    if coords.size == 0:
        return g.copy()
    r0, c0 = coords.min(axis=0)
    r1, c1 = coords.max(axis=0) + 1
    return g[r0:r1, c0:c1].copy()

PRIMITIVES = {
    'identity': identity,
    'rot90': rot90,
    'rot180': rot180,
    'rot270': rot270,
    'flip_h': flip_h,
    'flip_v': flip_v,
    'transpose': transpose,
    'crop_nonzero': crop_nonzero,
}

def learn_color_map(train_pairs):
    mapping = {}
    for inp, out in train_pairs:
        if inp.shape != out.shape:
            return None
        for c_in, c_out in zip(inp.flat, out.flat):
            c_in, c_out = int(c_in), int(c_out)
            if c_in in mapping and mapping[c_in] != c_out:
                return None
            mapping[c_in] = c_out
    return mapping

def apply_color_map(g, mapping):
    out = g.copy()
    for c_in, c_out in mapping.items():
        out[g == c_in] = c_out
    return out

def evaluate_program(fn, train_pairs):
    scores = []
    for inp, target in train_pairs:
        try:
            pred = fn(inp)
        except Exception:
            return 0.0
        if pred.shape != target.shape:
            scores.append(0.0)
        else:
            scores.append(float(np.mean(pred == target)))
    return float(np.mean(scores))

def synthesize_program(train_pairs):
    candidates = []
    for name1, fn1 in PRIMITIVES.items():
        candidates.append((name1, fn1, evaluate_program(fn1, train_pairs)))
    for name1, fn1 in PRIMITIVES.items():
        for name2, fn2 in PRIMITIVES.items():
            if name1 == 'identity' or name2 == 'identity':
                continue
            comp_fn = lambda g, f1=fn1, f2=fn2: f2(f1(g))
            candidates.append((f'{name2}({name1})', comp_fn, evaluate_program(comp_fn, train_pairs)))
    for name1, fn1 in PRIMITIVES.items():
        transformed = [(fn1(inp), out) for inp, out in train_pairs]
        cmap = learn_color_map(transformed)
        if cmap is not None:
            comp_fn = lambda g, f1=fn1, cm=cmap: apply_color_map(f1(g), cm)
            candidates.append((f'color_map({name1})', comp_fn, evaluate_program(comp_fn, train_pairs)))
    candidates.sort(key=lambda x: x[2], reverse=True)
    return candidates

def solve_task(task):
    train_pairs = [(np.array(p['input'], dtype=int), np.array(p['output'], dtype=int)) for p in task['train']]
    ranked = synthesize_program(train_pairs)
    fn1 = ranked[0][1]
    fn2 = ranked[1][1] if len(ranked) > 1 else fn1
    preds = []
    for item in task['test']:
        inp = np.array(item['input'], dtype=int)
        preds.append({'attempt_1': fn1(inp).tolist(), 'attempt_2': fn2(inp).tolist()})
    return preds

In [ ]:
# Execução em todas as tasks de teste e geração do submission.json
if TEST_CHALLENGES_PATH.exists():
    with open(TEST_CHALLENGES_PATH, 'r') as f:
        challenges = json.load(f)
    submission = {}
    for task_id, task in challenges.items():
        submission[task_id] = solve_task(task)
    with open(SUBMISSION_PATH, 'w') as f:
        json.dump(submission, f)
    print(f'Salvo submission.json com {len(submission)} tarefas resolvidas!')
else:
    print('Arquivo de teste do Kaggle não encontrado localmente. Rodando teste de verificação...')
    demo = {'train': [{'input': [[1, 0], [0, 1]], 'output': [[0, 2], [2, 0]]}], 'test': [{'input': [[1, 1], [0, 0]]}]}
    print('Predição demo:', solve_task(demo))